# ts2img: time series chart 실험

- **A (Colab) 또는 B (서버) 중 하나만** 실행한 뒤 1번부터 진행합니다.
- `%ts2img <명령>` 은 터미널의 `python -m ts2img <명령>` 과 같습니다.
- 끊겨도 같은 명령을 다시 실행하면 끝난 문항은 건너뛰고 이어서 돕니다.
- API key 는 셀에 적지 않습니다. `secrets.env` 또는 Colab Secrets(열쇠 아이콘)에 넣습니다.

## A. Colab 준비

런타임 유형은 T4 GPU. 결과는 Google Drive(`MyDrive/ts2img_results`)에 2분마다 복사되고, 다음 세션에서 자동으로 복원됩니다.
런타임이 꺼지면 이 섹션을 다시 실행한 뒤 같은 `run` 명령을 실행하세요.

In [ ]:
import os, sys
BRANCH = "perception_ablation"  # applied lab 실험이 있는 브랜치
if not os.path.isdir("/content/TStoImage"):
    !git clone -q -b {BRANCH} https://github.com/cv-team3-2026/TStoImage.git /content/TStoImage
%cd /content/TStoImage
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!"{sys.executable}" -m pip install -q -r requirements.txt

In [ ]:
from ts2img import notebook as nb
nb.setup(drive=True)

In [ ]:
%ts2img doctor

## B. 서버 준비

README 대로 clone, venv, `pip install` 을 마치고, 그 venv 를 kernel 로 이 노트북을 엽니다.
터미널이 편하면 노트북 없이 README 의 CLI 를 쓰면 됩니다. `--background` 로 실행하면 SSH 가 끊겨도 계속 돕니다.

In [ ]:
from ts2img import notebook as nb
nb.setup()

In [ ]:
%ts2img doctor

## A2. Applied 실험 (따릉이 · 다른 PC / Colab)

대형 VLM(예: Qwen-VL 230B)으로 **baseline만 넘는지** 볼 때 쓰는 두 실험입니다. 둘 다 `text` / `vision` / `text+vision` 을 돌립니다.

| 실험 | 과제 | 차트 |
| --- | --- | --- |
| `applied_seoul_peak_lab` | 내일 ≥1000대/시 피크 시각 집합 (F1), 60문항 | `bike_peak` ×3 |
| `applied_seoul_forecast_lab` | 다음 24h 평균 5단계, 90문항 | `line_style` ×3 |

**데이터:** UCI Seoul Bike. git 에 없고, 아래 셀이 zip 을 받아 `data/applied/seoul_bike/` 에 풉니다. 상세는 `ts2img/benchmarks/applied/README.md`.

**모델:** 기본은 `lab-vlm`. 다른 서버면 `secrets.env`(또는 Colab Secrets)에 `LAB_VLM_BASE_URL`, 필요 시 `LAB_VLM_API_KEY` 를 넣고 `configs/models.yaml` 의 `lab-vlm.model` 을 서버 모델 이름과 맞춥니다. 로컬 4B 시험은 `--models qwen3-vl-4b --auto-serve`.

In [ ]:
# Seoul Bike CSV 받기 (있으면 건너뜀). peak 도 같은 파일을 쓴다.
%ts2img download applied_seoul applied_seoul_peak

In [ ]:
EXP = "applied_seoul_peak_lab"  # 또는 applied_seoul_forecast_lab

In [ ]:
# 다른 PC 의 OpenAI-compatible 서버를 쓸 때 (LAB_VLM_* 필요)
%ts2img run $EXP --background

# 로컬 4B 로 짧게 확인
# %ts2img run $EXP --models qwen3-vl-4b --limit 5 --auto-serve

# 두 번째 실험
# %ts2img run applied_seoul_forecast_lab --background

## 1. 실험 실행

실험 = `configs/experiments/<이름>.yaml`. 새 chart 를 넣었다면 부록에서 그림을 먼저 확인하세요.

| 옵션 | 뜻 |
| --- | --- |
| `--auto-serve` | 로컬 vLLM 을 띄우고 끝나면 끈다 (처음에는 설치 포함 약 10분) |
| `--background` | 셀/SSH 가 끊겨도 계속 돈다. Colab 런타임이 꺼지면 같이 꺼진다 |
| `--limit N` | task 마다 앞에서 N 문항만 |
| `--models a,b` | yaml 의 모델 대신 (`configs/models.yaml` 의 key) |
| `--benchmarks x` | 대상 task 만 바꿔서. 예: `tsrbench/perception/anomaly_detection` |
| `--modalities x` | `text`, `vision`, `text+vision` 중 쉼표로 |
| `--repeats N` | 같은 설정 반복 횟수 |
| `--dry-run` | 실행하지 않고 남은 문항만 보여 준다 |

`smoke` 는 가짜 모델로 몇 초 만에 끝나는 설치 확인용입니다.

In [ ]:
EXP = "smoke"    # perception_baseline | perception_augment | tsrbench_full_baseline

In [ ]:
%ts2img run $EXP

# 예시
# %ts2img run perception_baseline --limit 20 --repeats 1 --auto-serve
# %ts2img run perception_augment --auto-serve --background
# %ts2img run perception_augment --models qwen3-vl-2b --modalities vision --auto-serve
# %ts2img run tsrbench_full_baseline --limit 20 --dry-run

## 2. 진행 확인 / 중지

In [ ]:
%ts2img status $EXP
# %ts2img status $EXP --watch    # 끝날 때까지 갱신
# %ts2img stop $EXP              # 중지. 다시 run 하면 이어서 돈다

## 3. 결과 보기

실행 중에도 만들 수 있습니다. `run` 에 `--limit`, `--models` 등을 줬다면 `report` 에도 같은 옵션을 주세요.
파일 위치: `results/reports/<실험>/report.html`

In [ ]:
%ts2img report $EXP
nb.show_report(EXP)

In [ ]:
nb.download_results("reports")    # Colab: 보고서 zip 다운로드. "all" 이면 raw 결과까지

## 부록. chart / prompt 미리보기 (선택)

모델을 부르지 않고 그림과 최종 입력만 확인합니다. chart 종류와 옵션: `%ts2img list`

In [ ]:
nb.show_charts(exp=EXP, n=3)
# nb.show_charts(["tsrbench_baseline", {"use": "line_style", "linewidth": 4}], task="tsrbench/perception/anomaly_detection", n=4)

In [ ]:
nb.show_prompt(task="tsrbench/perception", modality="vision")
# nb.show_prompt(exp=EXP)